# RAG Faithfulness

This notebook computes the Faithfulness score for some example RAG answers. 

1. Build RAG Chain
2. Simple Query: Compute faithfulness for a simple query containing just one relational information hop.
3. Complex Query: Compute faithfulness for a complex query involving more than one relational information hop. 

In [1]:
import logging

from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO)

# Vectorstore files were persisted here in notebook 1.vectorstore.ipynb
PERSIST_DIR = ".data/vectorstore"
COLLECTION_NAME = "arabidopsis_abstracts"
RAG_MODEL = "claude-sonnet-4-6"
FAITHFULNESS_MODEL = "claude-sonnet-4-6"

### 1. Build RAG Chain

In [2]:
from llm_knowledge_discovery.vectorstore import load_vectorstore
from llm_knowledge_discovery.rag import build_rag_chain

vectorstore = load_vectorstore(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

chain = build_rag_chain(
    vectorstore=vectorstore,
    model=RAG_MODEL,
    retrieval_k=10,
    rerank_k=5
)

print(f"RAG Chain:\n{chain}")

/Users/dylanelliott/workspace/llm-knowledge-discovery/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Loading vectorstore: collection='arabidopsis_abstracts', persist_dir='.data/vectorstore'
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Vectorstore loaded from '.data/vectorstore'
INFO:llm_knowledge_discovery.rag.chain:[chain.py] RAG chain built: model='claude-sonnet-4-6', retrieval_k=10, rerank_k=5


RAG Chain:
first={
  context: RunnableLambda(lambda q: retrieve_and_rerank(q, vectorstore, retrieval_k, rerank_k))
           | RunnableLambda(_format_context),
  question: RunnablePassthrough()
} middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="You are a plant biology research assistant. Answer the user's question using ONLY the provided abstracts. If the abstracts do not contain enough information to answer the question, say so and do not include a references section. If you can answer the question, select only the abstracts you actually use, renumber them sequentially starting from [1], cite them inline by their new number (e.g. [1], [2]), and include a References section at the end listing only the abstracts you cited in that same sequential order, using their exact titles as they appear in the conte

### 2. Simple Query

In [3]:
query = "What genes regulate flowering time in Arabidopsis?"

rag_result = chain.invoke(query)

print(f"Query:\n{query}\n")
print(f"Answer:\n{rag_result.answer}\n")
print(f"References:\n{rag_result.references}\n")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Query:
What genes regulate flowering time in Arabidopsis?

Answer:
Multiple genes and transcription factors have been identified as regulators of flowering time in Arabidopsis thaliana:

1. **CONSTANS (CO)**: A central gene in the photoperiodic pathway that activates the expression of the florigen FLOWERING LOCUS T (FT) in the leaves at the end of a long day [4]. CO expression is closely regulated by day length and modulated by both environmental and endogenous cues [1]. CO also upregulates key circadian clock genes such as CCA1, LHY, PRR5, and GI, suggesting a feedback loop between the photoperiod pathway and the circadian clock [4].

2. **FLOWERING LOCUS T (FT)**: A key florigen gene whose expression is regulated by multiple upstream factors. It is suppressed by BZR1 [1], repressed by IDD transcription factors (IDD14, IDD15, IDD16) [2], and regulated by TCP transcription factors through the photoperiod pathway [5].

3. **BRASSINAZOLE RESISTANT 1 (BZR1)**: A transcription factor in th

In [4]:
from llm_knowledge_discovery.rag import retrieve_and_rerank
from llm_knowledge_discovery.eval import score_faithfulness

context_docs = retrieve_and_rerank(query, vectorstore)

faithfulness_result = score_faithfulness(
    query=query,
    answer=rag_result.answer,
    context_docs=context_docs,
    model=FAITHFULNESS_MODEL
)

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.eval.faithfulness:[faithfulness.py] Extracted 47 claims from answer
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTT

In [5]:
dict(faithfulness_result)

{'query': 'What genes regulate flowering time in Arabidopsis?',
 'answer': 'Multiple genes and transcription factors have been identified as regulators of flowering time in Arabidopsis thaliana:\n\n1. **CONSTANS (CO)**: A central gene in the photoperiodic pathway that activates the expression of the florigen FLOWERING LOCUS T (FT) in the leaves at the end of a long day [4]. CO expression is closely regulated by day length and modulated by both environmental and endogenous cues [1]. CO also upregulates key circadian clock genes such as CCA1, LHY, PRR5, and GI, suggesting a feedback loop between the photoperiod pathway and the circadian clock [4].\n\n2. **FLOWERING LOCUS T (FT)**: A key florigen gene whose expression is regulated by multiple upstream factors. It is suppressed by BZR1 [1], repressed by IDD transcription factors (IDD14, IDD15, IDD16) [2], and regulated by TCP transcription factors through the photoperiod pathway [5].\n\n3. **BRASSINAZOLE RESISTANT 1 (BZR1)**: A transcripti

### 3. Complex Query

In [6]:
query = "What sort of single knock-out or over-expression strategies could potentially increase seed size and weight in Arabidopsis?"

rag_result = chain.invoke(query)

print(f"Query:\n{query}\n")
print(f"Answer:\n{rag_result.answer}\n")
print(f"References:\n{rag_result.references}\n")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Query:
What sort of single knock-out or over-expression strategies could potentially increase seed size and weight in Arabidopsis?

Answer:
Based on the provided abstracts, the following single knock-out or over-expression strategies could potentially increase seed size and weight in Arabidopsis:

1. **Knock-out of TCP4 (or related CIN-like TCP transcription factors):** The triple mutant *tcp3/4/10* produces enlarged seeds due to delayed endosperm cellularization and accelerated seed coat growth. TCP4 directly represses key factors for endosperm growth — *MINISEED3 (MINI3)*, *SHORT HYPOCOTYL UNDER BLUE1 (SHB1)*, and *AINTEGUMENTA (ANT)* — all of which promote seed and seed coat growth. While the evidence presented uses a triple mutant, disruption of TCP4 function is highlighted as a central mechanism, making it a strong candidate for a single knock-out strategy to increase seed size [1, 3].

2. **Over-expression of MINI3 or SHB1:** Since TCP4 directly represses *MINI3* and *SHB1* — key

In [7]:
context_docs = retrieve_and_rerank(query, vectorstore)

faithfulness_result = score_faithfulness(
    query=query,
    answer=rag_result.answer,
    context_docs=context_docs,
    model=FAITHFULNESS_MODEL
)

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.eval.faithfulness:[faithfulness.py] Extracted 23 claims from answer
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTT

In [8]:
dict(faithfulness_result)

{'query': 'What sort of single knock-out or over-expression strategies could potentially increase seed size and weight in Arabidopsis?',
 'answer': 'Based on the provided abstracts, the following single knock-out or over-expression strategies could potentially increase seed size and weight in Arabidopsis:\n\n1. **Knock-out of TCP4 (or related CIN-like TCP transcription factors):** The triple mutant *tcp3/4/10* produces enlarged seeds due to delayed endosperm cellularization and accelerated seed coat growth. TCP4 directly represses key factors for endosperm growth — *MINISEED3 (MINI3)*, *SHORT HYPOCOTYL UNDER BLUE1 (SHB1)*, and *AINTEGUMENTA (ANT)* — all of which promote seed and seed coat growth. While the evidence presented uses a triple mutant, disruption of TCP4 function is highlighted as a central mechanism, making it a strong candidate for a single knock-out strategy to increase seed size [1, 3].\n\n2. **Over-expression of MINI3 or SHB1:** Since TCP4 directly represses *MINI3* and